**Launch**  
Launch a fresh instance of Isaac Sim, go the top Menu Bar and click Window > Script Editor.

**Add a Ground Plane**

To add a ground plane using the interactive Python, copy paste the following snippet in the Script Editor and run it by clicking the Run button on the bottom.

In [ ]:
from isaacsim.core.api.objects.ground_plane import GroundPlane
GroundPlane(prim_path="/World/GroundPlane", z_position=0)

**Add a Light Source**  
You can add a light source to the scene to illuminate the objects in the scene. If you have a light source in the scene, but no object to reflect the light, the scene will still be dark.
1.   Open a new tab in the Script Editor (**Tab > Add Tab**).

2.   Add a light source by copy-pasting the following snippet in the Script Editor and running it.



In [ ]:
import omni.usd
from pxr import Sdf, UsdLux

stage = omni.usd.get_context().get_stage()
distantLight = UsdLux.DistantLight.Define(stage, Sdf.Path("/DistantLight"))
distantLight.CreateIntensityAttr(300)

**Add a Visual Cube**  
A “visual” cube is a cube with no physics properties attached. No mass, no collision. This cube will not fall under gravity or collide with other objects. You can press **Play** to see that the cube does not do anything when the simulation is running.

1.   Open a new tab in the Script Editor (**Tab > Add Tab**).
2.   Add two cubes by copy-pasting the following snippet in the Script Editor and run it. We’ll keep one as visual-only, and add physics and collision properties to the other for comparison.


In [ ]:
import numpy as np
from isaacsim.core.api.objects import VisualCuboid
VisualCuboid(
   prim_path="/visual_cube",
   name="visual_cube",
   position=np.array([0, 0.5, 0.5]),
   size=0.3,
   color=np.array([255, 255, 0]),
)
VisualCuboid(
   prim_path="/test_cube",
   name="test_cube",
   position=np.array([0, -0.5, 0.5]),
   size=0.3,
   color=np.array([0, 255, 255]),
)

Isaac Sim Core API are wrappers for raw USD and physics engine APIs. You can add a visual cube (without physics and color properties) using raw USD API. Notice that the raw USD API is more verbose, but gives you more control over each property.

In [ ]:
from pxr import UsdPhysics, PhysxSchema, Gf, PhysicsSchemaTools, UsdGeom
import omni

# USD api for getting the stage
stage = omni.usd.get_context().get_stage()

# Adding a Cube
path = "/visual_cube_usd"
cubeGeom = UsdGeom.Cube.Define(stage, path)
cubePrim = stage.GetPrimAtPath(path)
size = 0.5
offset = Gf.Vec3f(1.5,-0.2,1.0)
cubeGeom.CreateSizeAttr(size)
if not cubePrim.HasAttribute("xformOp:translate"):
  UsdGeom.Xformable(cubePrim).AddTranslateOp().Set(offset)
else:
  cubePrim.GetAttribute("xformOp:translate").Set(offset)

**Add Physics and Collision Properties**  
Common physics properties are mass and inertia matrix, which are the properties that allow the object to fall under gravity. Collision Properties are the properties that allow the object to collide with other objects.

Physics and collision properties can be added separately, so that you can have an object that collides with other objects but does not fall under gravity, or falls under gravity but does not collide with other objects. But in many cases, they are added together.

In Isaac Sim core API, we’ve written wrappers for frequently used objects with all its physics and collision properties attached. You can add a cube with physics and collision properties using the following snippet.

In [ ]:
import numpy as np
from isaacsim.core.api.objects import DynamicCuboid

DynamicCuboid(
   prim_path="/dynamic_cube",
   name="dynamic_cube",
   position=np.array([0, -1.0, 1.0]),
   scale=np.array([0.6, 0.5, 0.2]),
   size=1.0,
   color=np.array([255, 0, 0]),
)

Alternatively, if you want to modify an existing object to have physics and collision properties, you can use the following snippet.

In [ ]:
from isaacsim.core.prims import RigidPrim
RigidPrim("/test_cube")

from isaacsim.core.prims import GeometryPrim
prim = GeometryPrim("/test_cube")
prim.apply_collision_apis()

Click the **Play** button to see the cubes fall under gravity and collide with the ground plane.

**Move, Rotate, and Scale the Cube**

Moving an object using core API:

In [ ]:
import numpy as np
from isaacsim.core.prims import XFormPrim


translate_offset = np.array([[1.5,1.2,1.0]])
orientation_offset = np.array([[0.7,0.7,0,1]])     # note this is in radians
scale = np.array([[1,1.5,0.2]])

stage = omni.usd.get_context().get_stage()
cube_in_coreapi = XFormPrim(prim_paths_expr="/test_cube")
cube_in_coreapi.set_world_poses(translate_offset, orientation_offset)
cube_in_coreapi.set_local_scales(scale)

Moving an object using raw USD API:

In [ ]:
from pxr import UsdGeom, Gf
import omni.usd

stage = omni.usd.get_context().get_stage()
cube_prim = stage.GetPrimAtPath("/visual_cube_usd")
translate_offset = Gf.Vec3f(1.5,-0.2,1.0)
rotate_offset = Gf.Vec3f(90,-90,180)        # note this is in degrees
scale = Gf.Vec3f(1,1.5,0.2)
# translation
if not cube_prim.HasAttribute("xformOp:translate"):
   UsdGeom.Xformable(cube_prim).AddTranslateOp().Set(translate_offset)
else:
   cube_prim.GetAttribute("xformOp:translate").Set(translate_offset)
# rotation
if not cube_prim.HasAttribute("xformOp:rotateXYZ"):        # there are also "xformOp:orient" for quaternion rotation, as well as "xformOp:rotateX", "xformOp:rotateY", "xformOp:rotateZ" for individual axis rotation
   UsdGeom.Xformable(cube_prim).AddRotateXYZOp().Set(rotate_offset)
else:
   cube_prim.GetAttribute("xformOp:rotateXYZ").Set(rotate_offset)
# scale
if not cube_prim.HasAttribute("xformOp:scale"):
   UsdGeom.Xformable(cube_prim).AddScaleOp().Set(scale)
else:
   cube_prim.GetAttribute("xformOp:scale").Set(scale)

Save your work.